##List of Homework's exercises:

1.   [Ex1](#scrollTo=ux5PBYkbwewj) - 2 points
2.   [Ex2](#scrollTo=Z5C3GX9eXFF_) - 4 points
3.   [Ex3](#scrollTo=uHQscVCD7rM0) - 4 points
4.   [Bonus 1](#scrollTo=iJeG56t9-jWI) - 1 point
5.   [Bonus 2](#scrollTo=cLoHOqAcu0Q9) - up to 3 points
6.   [Bonus 3](#scrollTo=jdZkblZW7bEp) - up to 3 points (based on quality of presentation)




---


## Homework exercise 1: implement and train a bagging classifier with 3 K-NN models as estimators (2 points)


<font color='red'> In this exercise you will need to use `classify_knn` function from the first practice session to train three different KNN models on three resamples of this dataset. </font>


# Homework #5

## Ensemble learning

This colaboratory contains Homework #5 of the Machine Learning course, which is due **April 4, midnight (23:59 EEST time)**. To complete the homework, extract **(File -> Download .ipynb)** and submit to the course webpage.


## Submission's rules:

1.   Please, submit only .ipynb that you extract from the Colaboratory.
2. Run your homework exercises before submitting (output should be present, preferably restart the kernel and press run all the cells).
3. Do not change the description of tasks in red (even if there is a typo|mistake|etc).
4. Please, make sure to avoid unnecessary long printouts.
5. Each task should be solved right under the question of the task and not elsewhere.
6. Solutions to both regular and bonus exercises should be submitted in one IPYNB file.

Please, steer clear of copying someone else's work. If you discuss assignments with anyone in the course, please, mention their names here:

Pooh

In [1]:
import pandas as pd
import numpy as np

In [2]:
def create_random_2c_data (D, N):
  """
  Function create_random_2c_data generates two sets of D dimensional
  points (N points each), one for each class. The first set is sampled from D
  dimensional Gaussian distribution with mean 0 and standard deviation 1. The
  second set is generated from the distribution, with mean 1 and standard
  deviation 1.
  """
  # Generating N points for the first class
  mu_vec1 = np.zeros(D) # creates a vector of zeros, these are averages across each dimension
  cov_mat1 = np.eye(D) # creates a diagonal matrix of size D x D, all values except diagonal are 0
  class1_sample = np.random.multivariate_normal(mu_vec1, cov_mat1, N)

  # The same stuff as above, just averages are shifted into 1
  mu_vec2 = np.ones(D) # creates a vector of ones
  cov_mat2 = np.eye(D)
  class2_sample = np.random.multivariate_normal(mu_vec2, cov_mat2, N)

  # a lot of boring things....
  # gluing together two matrices generated above
  data = pd.DataFrame(np.concatenate((class1_sample, class2_sample)))

  # Create names for columns
  data.columns = [ 'x' + str(i) for i in (np.arange(D)+1)]

  # Create a class column
  data['class'] = np.concatenate((np.repeat(0, N), np.repeat(1, N)))

  # This is important for plotting and modelling
  data['class'] = data['class'].astype('category')

  return data

In [3]:
from sklearn.model_selection import train_test_split
from sklearn.utils import resample
np.random.seed(2342347823) # random seed, this number was random, no need to build conspiracies around it

D = 2 # two dimensions
N = 100 # points per class

whole_data = create_random_2c_data(D, N)

# Randomly splitting data into train (60%) and validation (40%)
train, val = train_test_split(whole_data, random_state = 111, test_size = 0.40)

n_bootstraps = 3
np.random.seed(1111)

# creating resamples
resamples = [resample(train, n_samples = len(train), replace=True).index.values for i in range(n_bootstraps)]

# first resample
train_resample1 = train.loc[resamples[0]]

# second resample
train_resample2 = train.loc[resamples[1]]

# third resample
train_resample3 = train.loc[resamples[2]]

<font color='red'> Here, I just convert pandas DataFrame into Numpy arrays that are easier to use list comprehension mechanisms on. </font>

In [4]:
train1 = np.asarray(train_resample1[['x1','x2']])
labels1 = np.asarray(train_resample1[['class']]).reshape((train_resample1.shape[0]))

train2 = np.asarray(train_resample2[['x1','x2']])
labels2 = np.asarray(train_resample2[['class']]).reshape((train_resample2.shape[0]))

train3 = np.asarray(train_resample3[['x1','x2']])
labels3 = np.asarray(train_resample3[['class']]).reshape((train_resample3.shape[0]))

val_points = np.asarray(val[['x1','x2']])
val_labels = np.asarray(val[['class']]).reshape((val.shape[0]))

<font color='red'>  **(Homework exercise 1- a)** Copy and adapt `classify_knn` function from the first homework and practice session to operate on 2D points. **(0.5 points)**</font>

In [5]:
def dist(point1, point2): # function dist is also from the first practice session
  # sum of squared coordinate-wise differences under sqrt
  return(np.sqrt(np.sum((point2 - point1)**2)))

def classify_knn(val_point, k, train, labels):

  ##### YOUR CODE STARTS #####
  all_distances = np.apply_along_axis(dist, axis=1, arr=train, point2=val_point)
  nearest_neighbours = np.argsort(all_distances)[:k]
  predicted_classes = labels[nearest_neighbours]
  prediction = np.argmax(np.bincount(predicted_classes))
  ##### YOUR CODE ENDS #####

  return prediction

<font color='red'> Test that the function was adapted correctly by running the following example </font>

In [6]:
val_point = val_points[1]
print(f'predicted class of the first point is {classify_knn(val_point, 5,  train1, labels1)}, while the true class is {val_labels[1]}')

predicted class of the first point is 0, while the true class is 0


<font color='red'> **(Homework exercise 1- b)** Classify each point from the validation set using `classify_knn` function. Use different resamples and list comprehension. Fix `k` at 5. **(1 point)**</font>


In [7]:
k = 5

# Use three K-NN models that work on three different resamples

##### YOUR CODE STARTS #####
val['knn1'] = [classify_knn(p, k, train1, labels1) for p in val_points]
val['knn2'] = [classify_knn(p, k, train2, labels2) for p in val_points]
val['knn3'] = [classify_knn(p, k, train3, labels3) for p in val_points]
##### YOUR CODE ENDS #####

<font color='red'> Below aggregate individual predictions using the majority vote approach</font>

In [8]:
##### YOUR CODE STARTS #####
val['knn_bagging'] = val[['knn1', 'knn2', 'knn3']].mode(axis =1)
##### YOUR CODE ENDS #####

print(f"Accuracy of hand made bagged ensemble with 3 KNNs is {np.sum(val['knn_bagging'] == val['class'])/len(val[['class']])*100}%")

Accuracy of hand made bagged ensemble with 3 KNNs is 66.25%


Why not 67?

<font color='red'> **(Homework exercise 1- c)** Use sklearn `BaggingClassifier` to implement analogous model that uses KNeighborsClassifier as an estimator (with k = 5). Don't forget to use a random state for reproducibility.

Assess its performance on the same validation set and display it. **(0.5 points)**</font>


In [9]:
from sklearn.ensemble import BaggingClassifier
from sklearn.neighbors import KNeighborsClassifier

##### YOUR CODE STARTS #####
knn = KNeighborsClassifier(n_neighbors=5)
knn_begger = BaggingClassifier(estimator=knn, n_estimators=10, random_state=42)
knn_begger.fit(train[['x1','x2']], train['class'])
knn_begger.predict(val[['x1','x2']])
##### YOUR CODE ENDS #####
print(f"Accuracy of sklearn bagging with {3} KNNs {knn_begger.score(val[['x1', 'x2']], val[['class']])*100}%")

Accuracy of sklearn bagging with 3 KNNs 75.0%


## Homework exercise 2: eXtreme Gradient Boosting (XGBoost) (4 points)

<font color='red'> Let's finally build for ourselves a new shiny XGBoost model, the most popular algorithm for Kaggle competitions. </font>
<font color='red'>  Since XGBoost truly shines on tabular data, we are going to use the Thyroid Disease Dataset for training an XGboost model. Further details about this dataset can be found at
 [here](https://www.openml.org/search?type=data&status=active&id=40475). </font>


In [10]:
from sklearn.datasets import fetch_openml

data = fetch_openml(name="thyroid-allhyper")

In [11]:
#Let's take a look at data
data.data

,V1,V2,V3,V4,V5,V6,V7,V8,V9,V10,...,V17,V18,V19,V20,V21,V22,V23,V24,V25,V26
0,41.0,1,1,1,1,1,1,1,1,1,...,2,1.30000,2,2.500000,2,125.0,2,1.140000,2,109.000000
1,23.0,1,1,1,1,1,1,1,1,1,...,2,4.10000,2,2.000000,2,102.0,1,0.997912,1,110.787984
2,46.0,2,1,1,1,1,1,1,1,1,...,2,0.98000,1,2.024966,2,109.0,2,0.910000,2,120.000000
3,70.0,1,2,1,1,1,1,1,1,1,...,2,0.16000,2,1.900000,2,175.0,1,0.997912,1,110.787984
4,70.0,1,1,1,1,1,1,1,1,1,...,2,0.72000,2,1.200000,2,61.0,2,0.870000,2,70.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2795,70.0,2,1,1,1,1,1,1,1,1,...,2,2.70000,1,2.024966,2,155.0,2,1.050000,2,148.000000
2796,73.0,2,1,2,1,1,1,1,1,1,...,1,4.67215,2,0.700000,2,63.0,2,0.880000,2,72.000000
2797,75.0,2,1,1,1,1,1,1,1,1,...,1,4.67215,1,2.024966,2,147.0,2,0.800000,2,183.000000
2798,60.0,1,1,1,1,1,1,1,1,1,...,2,1.40000,1,2.024966,2,100.0,2,0.830000,2,121.000000


In [12]:
# Types of classes
data.target

0       3
1       1
2       1
3       1
4       5
       ..
2795    5
2796    1
2797    1
2798    1
2799    5
Name: Class, Length: 2800, dtype: category
Categories (5, object): ['1', '2', '3', '4', '5']

In [13]:
# Number of instances for each class
data.target.value_counts()

Class
1    1632
5     771
3     275
2      91
4      31
Name: count, dtype: int64

<font color='red'> **(Homework exercise 2- a)** Complete the preprocessing steps and train the XGBoost model for the Thyroid Disease classification task.
Find additional information regarding the training of an XGBoost model [here](https://xgboost.readthedocs.io/en/latest/python/python_intro.html) **(1.5 points)** </font>

In [38]:
# Initial data preparation, which involves spliting the dataset into training, validation, and test sets
# Use 80/10/10 datasplit

##### YOUR CODE STARTS #####
df = data.frame

for i in df.columns:
    if df[i].dtype == 'category':
        df[i] = df[i].astype('float64')
        
train, temp = train_test_split(df, random_state=111, test_size=0.2)
val, test = train_test_split(temp, random_state=111, test_size=0.5)

#we need, that our classes startes from 0, not 1
train['Class'] = train['Class'] - 1
val['Class']   = val['Class'] - 1
test['Class']  = test['Class'] - 1
##### YOUR CODE ENDS #####

In [39]:
import xgboost as xgb
from sklearn.metrics import accuracy_score

##### YOUR CODE STARTS #####

# XGBoost wants data to be wrapped into special formats
dtrain = xgb.DMatrix(train.drop(columns ='Class'), label = train['Class'])
dval = xgb.DMatrix(val.drop(columns ='Class'), label = val['Class'])
dtest = xgb.DMatrix(test.drop(columns ='Class'), label = test['Class'])

# Define the XGBoost parameters (you can add more parameters if you want)
params = {
    "objective": "multi:softmax",
    "num_class": 5,
    "eval_metric": "mlogloss",
}

# Train for training and val for validation
eval_list = [(dtrain, "train"), (dval, "val")]

# Train the XGBoost model
model = xgb.train(
    params,
    dtrain,
    num_boost_round = 200,
    evals = eval_list,
    early_stopping_rounds = 10
)

##### YOUR CODE ENDS #####

[0]	train-mlogloss:0.90137	val-mlogloss:0.87078
[1]	train-mlogloss:0.81711	val-mlogloss:0.79744
[2]	train-mlogloss:0.75336	val-mlogloss:0.74658
[3]	train-mlogloss:0.70537	val-mlogloss:0.71128
[4]	train-mlogloss:0.67072	val-mlogloss:0.68609
[5]	train-mlogloss:0.63994	val-mlogloss:0.66583
[6]	train-mlogloss:0.61681	val-mlogloss:0.65000
[7]	train-mlogloss:0.59646	val-mlogloss:0.63654
[8]	train-mlogloss:0.57846	val-mlogloss:0.62753
[9]	train-mlogloss:0.56172	val-mlogloss:0.62094
[10]	train-mlogloss:0.54300	val-mlogloss:0.61748
[11]	train-mlogloss:0.53060	val-mlogloss:0.61318
[12]	train-mlogloss:0.51505	val-mlogloss:0.60403
[13]	train-mlogloss:0.50266	val-mlogloss:0.60578
[14]	train-mlogloss:0.49384	val-mlogloss:0.60373
[15]	train-mlogloss:0.48526	val-mlogloss:0.59969
[16]	train-mlogloss:0.47760	val-mlogloss:0.59711
[17]	train-mlogloss:0.46569	val-mlogloss:0.60231
[18]	train-mlogloss:0.46035	val-mlogloss:0.60218
[19]	train-mlogloss:0.45409	val-mlogloss:0.60159
[20]	train-mlogloss:0.44819	va

<font color='red'> **(Homework exercise 2- b)**  Make predictions on the test data and evaluate the model **(0.5 points)** </font>

In [43]:
##### YOUR CODE STARTS #####
prediction = model.predict(dtest)

prediction = prediction + 1
y_test = test['Class'] + 1

accuracy = accuracy_score(y_test, prediction)
accuracy
##### YOUR CODE ENDS #####

0.7035714285714286

<font color='red'> **(Homework exercise 2- c)**  In machine learning, feature importance scores are used to determine the relative importance of each feature in a dataset when building a predictive model. Such scores can be derived from a variety of different models, including ensembles. Explore feature importance of the resulting XGBoost model and report the top five most important features **(0.5 points)** </font>



In [46]:
##### YOUR CODE STARTS #####
importance_dict = model.get_score(importance_type='gain')
importance_dict
##### YOUR CODE ENDS #####

{'V1': 1.5747485160827637,
 'V2': 1.7152706384658813,
 'V3': 4.064698219299316,
 'V4': 0.8039936423301697,
 'V5': 1.5263022184371948,
 'V6': 4.407073497772217,
 'V7': 1.1016863584518433,
 'V9': 3.0196855068206787,
 'V10': 1.1079630851745605,
 'V11': 0.9495410323143005,
 'V12': 4.449991703033447,
 'V13': 1.229327917098999,
 'V14': 0.3985675275325775,
 'V16': 17.59255599975586,
 'V17': 3.84440016746521,
 'V18': 1.246701717376709,
 'V19': 12.876396179199219,
 'V20': 2.604377269744873,
 'V21': 0.5218021869659424,
 'V22': 1.1488978862762451,
 'V23': 5.7380290031433105,
 'V24': 1.590134859085083,
 'V26': 1.028973937034607}

<font color='red'> **(Homework exercise 2- d)** Train Adaptive Boosting, Gradient Boosting and a simple KNN model from sklearn (KNeighborsClassifier) on the same training data and evaluate on the same test data. For each model use the default hyperparameters (e.g. `n_estimators` or `n_neighbors`). If you do not want to use default parameters, you can use `cross_val_score` to pick the best values for the hyperparameters using training data. Compare the performance of these three models and XGBoost and draw conclusions in a separate text cell  **(1.5 points)** </font>

In [ ]:
# AdaBoostClassifier
%%time
##### YOUR CODE STARTS #####
...
##### YOUR CODE ENDS #####

In [ ]:
# GradientBoostingClassifier
%%time
# might take considerable time if trained with default number of n_estimators
##### YOUR CODE STARTS #####
...
##### YOUR CODE ENDS #####

In [ ]:
# KNeighborsClassifier
%%time
##### YOUR CODE STARTS #####
...
##### YOUR CODE ENDS #####

<font color='red'> How these models compare with each other and to XGBoost? Can you try to elaborate on this difference? </font>

In [ ]:
# Write your comment here:


## Homework exercise 3: Kaggle in-class to predict whether a patient will be readmitted to the hospital within 30 days after discharge (4 points)
<font color='red'> In this exercise we will see how well ensembles work in the real-life problems. Here we are going to work with medical data, and more specifically we will attempt to predict whether a patient will be readmitted to the hospital within 30 days after discharge based on numerical and categorical features. To this end, we have created a corresponding Kaggle in-class competition: https://www.kaggle.com/t/cfc5280da3b847ab8314140f3b5fd61a (this is invitation link). </font>

<font color='red'> **(Homework exercise 3- a)** Getting started: follow the code provided with the Kaggle competition (`sample_notebook.ipynb`) to make the first sample submission using `random_predict` function, Kaggle API or by manually uploading submission file to https://www.kaggle.com/competitions/ml-kse-2026. Report your score. **(0.5 points)** </font>

In [ ]:
import pandas as pd
import numpy as np
import random

<font color='red'> Load `train.csv` and `test.csv` from https://www.kaggle.com/competitions/ml-kse-2026

In [ ]:
from google.colab import files
files.upload();

In [ ]:
files.upload();

In [ ]:
##### YOUR CODE STARTS #####
...
##### YOUR CODE ENDS #####

<font color='red'> Insert your random guesser below:

In [ ]:
##### YOUR CODE STARTS #####
...
##### YOUR CODE ENDS #####

<font color='red'> Create a sample submission below

In [ ]:
##### YOUR CODE STARTS #####
...
##### YOUR CODE ENDS #####

<font color='red'> You can either use Kaggle API below to submit submission file or download file and submit it manually to Kaggle via web interface. </font>

<font color='red'> You need to have an account on Kaggle.com, before you proceed. When you access your kaggle profile, you need to download your API Token from kaggle. It's very easy:
1. Click on your profile icon
2. Go to **Account**
3. In **API** you press **Create Legacy API token**

**Optionally:** You can create not a Legacy type, altough the token preparation in the notebook process will differ

<font color='red'> Now we load the file **kaggle.json** that you have downloaded, into this notebook: </font>

In [ ]:
files.upload(); # upload your kaggle.json file

<font color='red'> The next cell moves the file into a separate folder, sets secure access for it and configures your Kaggle profile for this notebook.</font>

In [ ]:
import json

!mkdir /root/.kaggle/
!mv kaggle.json /root/.kaggle/kaggle.json
!chmod 600 ~/.kaggle/kaggle.json
!kaggle config set -n path -v{/content}

<font color='red'> Assuming that you have accepted the rules of the competition, you should be able to make a submission using the following code: </font>

In [ ]:
!kaggle competitions submit -c ml-kse-2026 -f sample_submission.csv -m "Random guess submission"

<font color='red'> What was the score you received? </font>

In [ ]:
# Report your result here:


<font color='red'> **(Homework exercise 3- b)** Take off: make two more submissions with at least two different ensemble models we have studied. Make sure to perform all necessary data pre-processing steps to accommodate the requirements of these ensemble models (e.g. data imputation and restructuring). To get full points, shortly summarise your pre-processing, models you have selected and rational behind this choice, and finally report your scores. **(2 points)** </font>

In [ ]:
##### YOUR CODE STARTS #####
...
##### YOUR CODE ENDS #####

<font color='red'> Summarise your pre-processing steps, models selected (why these?): </font>

In [ ]:
# Add your comment here:


<font color='red'> **(Homework exercise 3- c)** Reaching superiority: tune the above models or train a new one (does not have to be ensemble) such that you beat Mr.Smart (`mr_smart.csv`) benchmark. If one or both of your models from 3-b are already superior to Mr.Smart, then you should beat them instead. In order to claim the point in this exercise, describe the model, data pre-processing steps and report your score (which should be higher than Mr Smart's and your previous' models). **(1.5 points)** </font>

In [ ]:
##### YOUR CODE STARTS #####
...
##### YOUR CODE ENDS #####

<font color='red'> Summarise your pre-processing steps, models selected (why these?), and the "special sauce" (what you changed compared to 3-b) below: </font>

In [ ]:
# Add your comment here:


# Bonus exercises
*(NB, these are optional exercises!)*


##Bonus exercise 1 (1 bonus point): Interim Champion

<font color='red'> If at any point of time your submission lended at the top of the **public leaderboard** from the Kaggle competition (Exercise 3), take a screenshot of this remarkable achievement before it is too late and somebody else did not knock it off. Post this screenshot below (add a picture). Describe your solution below (do not have to write too much, just the main idea). You can only get this point once.

In [ ]:
# Add your comment here:


## Bonus exercise 2 (up to 3 bonus points): Top Perfomer Challenge

<font color='red'> You are encouraged to compete for the top three places in the Kaggle competition described in Exercise 3. So, **3 points** for the first place, **2 points** for the second place and **1.5 points** for the third place.

<font color='red'> To receive the bonus points, you must finish in the top three of the **private leaderboard** and maintain your position throughout the competition. The end of the competition is 04.04.2026 23:59:59.</font>

## Bonus exercise 3 (up to 3 bonus points depending on presentation): Discrete AdaBoost from scratch



<font color='red'> In the class we have discussed a simplified version of the AdaBoost, the goal of this task is to implement an unabrdiged version of the Discrete AdaBoost classifier following the instructions from the paper [Friedman et al (2000)](https://hastie.su.domains/Papers/AdditiveLogisticRegression/alr.pdf). In a nutshell, AdaBoost classifier assigns initial weights to training examples, fits a weak classifier, and updates the data and classifier weights based on classification accuracy. The process is repeated for each weak estimator. The final strong classifier combines weak classifiers with different weights.</font>

<font color='red'> Implement AdaBoost classifier.

Tips:

- use `DecisionTreeClassifier` from `sklearn` as a weak classifier
- avoid division by zero
- pay attention to the format of data labels.</font>



In [ ]:
class AdaBoost:
  def __init__(self, n_estimators=50):
    self.n_estimators = n_estimators
    self.estimator_weights = []
    self.models = []

  def fit(self, X, y):
    ##### YOUR CODE STARTS #####
    ...
    ##### YOUR CODE ENDS #####

  def predict(self, X):
    ##### YOUR CODE STARTS #####
    ...
    ##### YOUR CODE ENDS #####



<font color='red'> Test your model on the dataset from exercise 3. Report accuracy on the test set and observe how the performance changes with the different number of estimators.</font>

In [ ]:
##### YOUR CODE STARTS #####
...
##### YOUR CODE ENDS #####

<font color='red'> Use sklearn implementation of AdaBoost classifier with the same parameters as you used for your implementation. Report accuracy on the test set as well. </font>

In [ ]:
##### YOUR CODE STARTS #####
...
##### YOUR CODE ENDS #####

<font color='red'> Report difference in results, if there are any. Explain why sklearn's model behaves differently, you might want to take a look at sklearn's [code](https://github.com/scikit-learn/scikit-learn/blob/093e0cf14/sklearn/ensemble/_weight_boosting.py#L341) :)</font>

<font color='red'> Your answer:</font>

# Comments (optional feedback to the course instructors)
Here, please, leave your comments regarding the practice session, possibly answering the following questions:

* how much time did you spend on this homework?
* was it too hard/easy for you?
* what would you suggest to add or remove?
* anything else you would like to tell us

Your comments:

# <font color='red'>  End of the homework. Please don't delete this cell.</font>